# Amazon ML Challenge 2026 — Business Entity Resolution (Kaggle runner)

This notebook **imports the dataset**, **imports the code from GitHub**, then **trains and predicts** with it,
and finally writes the files the challenge needs (`matching_results.tsv`, `candidate_pairs.tsv`, the submission zip).

### Before you run — notebook settings (right-hand panel)
1. **Accelerator:** `GPU T4 x2`
2. **Internet:** `On` (to clone the code and download the cross-encoder model; needs a phone-verified Kaggle account)
3. **Input:** *Add Input → Datasets → Your Datasets →* **`amazon-ml`** (the dataset you created from `amazon-ml.zip`;
   any folder layout inside the zip works, the 7 challenge `.tsv` files just have to be somewhere in it)

### How to run
* **Quick check first (recommended, ~10–15 min):** set `RUN_SMOKE_FIRST = True` and `RUN_FULL = False`, then *Run All*.
* **Full run:** set `RUN_FULL = True`, then **Save Version → Save & Run All (Commit)**. A commit keeps running after you
  close the browser (Kaggle limit: 12 h). Results appear in the version's **Output** tab.
* If a stage fails in an interactive session, fix the setting and **re-run from that cell** — finished stages are skipped.
* Running short on time? Put e.g. `"ingest.train_s1_frac=0.6"` in `EXTRA_OVERRIDES` (trains on 60% of the train clusters)
  or set `USE_CROSS_ENCODER = False`.

In [ ]:
# ---- settings -----------------------------------------------------------------------------------
REPO_URL = "https://github.com/Mveen3/amazon-ml-challange.git"
BRANCH = "main"
DATASET_SLUG = "amazon-ml"      # your Kaggle dataset name (its folder under /kaggle/input)
TEAM_NAME = "my_team"           # used for <team>_submission.zip
RUN_SMOKE_FIRST = False         # True: 5k-entity end-to-end check (incl. the real cross-encoder on GPU)
RUN_FULL = True                 # False: stop after setup (+ smoke test)
USE_CROSS_ENCODER = True        # False: skip the neural stage (saves ~1-1.5 h)
EXTRA_OVERRIDES = []            # any config value, e.g. ["ingest.train_s1_frac=0.6"]
CONFIG = "configs/kaggle.yaml"

import json, os, shutil, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "amazon-ml-challange"
PKG = REPO_DIR / "code" / "business_entity_resolution"
LOG_DIR = WORKING / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout[-6000:])
    if r.stderr:
        print(r.stderr[-3000:])
    if check and r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r

## 1. Hardware check

In [ ]:
sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv", check=False)
print("CPU cores:", os.cpu_count())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
except ImportError:
    pass
for p in ["/kaggle/working", "/kaggle/temp", "/tmp"]:
    try:
        os.makedirs(p, exist_ok=True)
        print(f"{p}: {shutil.disk_usage(p).free / 1e9:.0f} GB free")
    except OSError as e:
        print(p, e)
import torch
print("torch", torch.__version__, "| CUDA GPUs:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    print("WARNING: no GPU found - set Accelerator to 'GPU T4 x2' (CPU-only will be far slower).")

## 2. Import the code from GitHub

In [ ]:
if (REPO_DIR / ".git").exists():
    sh(f"git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD")
else:
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log -1 --format='%h %s (%cd)'")

## 3. Install the few packages Kaggle does not already have

In [ ]:
sh(f"{sys.executable} -m pip install -q -r {PKG}/requirements-kaggle.txt")
sh(f"{sys.executable} -c \"import polars, rapidfuzz, xgboost, transformers, sklearn; "
   f"print('polars', polars.__version__, '| rapidfuzz', rapidfuzz.__version__, '| xgboost', xgboost.__version__, "
   f"'| transformers', transformers.__version__, '| sklearn', sklearn.__version__)\"")

## 4. Import the dataset
Finds the 7 challenge files anywhere under `/kaggle/input/<DATASET_SLUG>` (extracting any zip/tar still packed)
and links them into `<scratch>/dataset/{train,test}/`, the layout the pipeline expects.

In [ ]:
sys.path.insert(0, str(PKG / "scripts"))
import kaggle_prepare

SCRATCH = kaggle_prepare.pick_scratch()          # large ephemeral disk (not saved as output)
DATA_DIR = SCRATCH / "dataset"
WORK_DIR = SCRATCH / "ber_work"
OUTPUT_DIR = REPO_DIR / "output"
kaggle_prepare.prepare("/kaggle/input", DATASET_SLUG, str(DATA_DIR))

## 5. Pipeline runner

In [ ]:
BASE_SETS = [f"paths.work_dir={WORK_DIR}", f"paths.data_dir={DATA_DIR}", f"paths.output_dir={OUTPUT_DIR}"]
if not USE_CROSS_ENCODER:
    BASE_SETS.append("ce.enabled=false")


def run(stages, config=CONFIG, sets=None, log_name="pipeline.log"):
    """Run pipeline stages in a subprocess, streaming the log here and into /kaggle/working/logs."""
    sets = BASE_SETS + list(EXTRA_OVERRIDES) if sets is None else sets
    cmd = [sys.executable, "-u", "-m", "ber.pipeline.run", "--config", config, "--stage", stages]
    for s in sets:
        cmd += ["--set", s]
    env = dict(os.environ, PYTHONPATH=str(PKG / "src"), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    print("$", " ".join(cmd))
    t0 = time.time()
    with open(LOG_DIR / log_name, "a") as log:
        p = subprocess.Popen(cmd, cwd=PKG, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        p.wait()
    print(f"\n[{stages}] exit {p.returncode} in {(time.time() - t0) / 60:.1f} min | "
          f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of 12")
    if p.returncode != 0:
        raise RuntimeError(f"stage(s) '{stages}' failed - see the log above ({LOG_DIR / log_name}). "
                           "Re-running this cell resumes at the failed stage.")

## 6. (Optional) smoke test — 5k entities, whole pipeline incl. the real cross-encoder on GPU
Scores are much higher than on the real data (the sample's distractors are random); it only proves everything runs.

In [ ]:
if RUN_SMOKE_FIRST:
    sd = SCRATCH / "smoke"
    sh(f"cd {PKG} && {sys.executable} scripts/make_sample.py --data {DATA_DIR} --out {sd}/sample_data")
    smoke_sets = [f"paths.data_dir={sd}/sample_data", f"paths.work_dir={sd}/work", f"paths.output_dir={sd}/output",
                  "blocking.knn.device=auto"]
    if USE_CROSS_ENCODER:
        smoke_sets += ["ce.enabled=true", "ce.model_name=intfloat/multilingual-e5-small", "ce.max_pos=3000",
                       "ce.batch_size=64"]
    run("all", config="configs/smoke.yaml", sets=smoke_sets, log_name="smoke.log")
    sh(f"cd {PKG} && {sys.executable} scripts/score_sample.py --out {sd}/output "
       f"--truth {sd}/sample_data/test_truth.tsv --s1 {sd}/sample_data/test/test_source1.tsv")

## 7. Full run
Each cell is one group of stages; the log shows progress and the session time used so far.

In [ ]:
if RUN_FULL:
    run("ingest,eda,mine,normalize")      # load TSVs, mine lookup tables from train pairs, parse every record

In [ ]:
if RUN_FULL:
    run("block")                          # candidate generation (TF-IDF/RP kNN on GPU + exact keys)

In [ ]:
if RUN_FULL:
    run("prerank,expand")                 # pre-ranker + floor tuned on the train ceiling -> candidate set

In [ ]:
if RUN_FULL:
    run("features")                       # round-1 pair features (CPU, all cores)

In [ ]:
if RUN_FULL:
    run("r1")                             # round-1 GBDT (XGBoost on GPU), OOF on train

In [ ]:
if RUN_FULL and USE_CROSS_ENCODER:
    run("ce_train,ce_infer")              # cross-encoder on the uncertain band (both T4s via DataParallel)

In [ ]:
if RUN_FULL:
    run("r2,gate,tune,predict,outputs")   # round 2, entity gate, thresholds, submission files + validator

## 8. Results

In [ ]:
if RUN_FULL:
    rep = json.load(open(WORK_DIR / "models" / "oof_report.json"))
    keys = ["macro_f05", "ceiling", "micro_precision", "micro_recall", "by_profile", "singleton_f05",
            "non_singleton_f05", "cands_per_s1"]
    print("Out-of-fold validation on the full train set (your best pre-submission estimate):")
    print(json.dumps({k: rep.get(k) for k in keys}, indent=1))
    reports = WORKING / "reports"
    reports.mkdir(exist_ok=True)
    for f in ["oof_report.json", "thresholds.json", "stress_check.json", "prerank/floor.json",
              "r1/oof_metrics.json", "r2/oof_metrics.json", "gate/oof_metrics.json"]:
        src = WORK_DIR / "models" / f
        if src.exists():
            shutil.copy(src, reports / f.replace("/", "_"))
    shutil.copy(OUTPUT_DIR / "matching_results.tsv", WORKING / "matching_results.tsv")
    shutil.copy(OUTPUT_DIR / "candidate_pairs.tsv", WORKING / "candidate_pairs.tsv")

## 9. Final submission package
Builds `<team>_submission.zip` in the required layout (`output/`, `code/business_entity_resolution/`,
`Documentation_template.md`), validates it, and writes `<team>_models.tar.gz` (trained models for the
inference-only reproduction path).

In [ ]:
if RUN_FULL:
    env = f"WORK_DIR={WORK_DIR} DATA_DIR={DATA_DIR} OUT_DIR={WORKING}"
    sh(f"cd {PKG} && {env} bash scripts/package_submission.sh {TEAM_NAME} --with-models")
    for p in sorted(WORKING.iterdir()):
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) if p.is_dir() else p.stat().st_size
        print(f"{size / 1e6:10.1f} MB  {p}")

### Download (version → **Output** tab)
* `matching_results.tsv` — upload this to the challenge portal (leaderboard file)
* `<team>_submission.zip` — the final package (fill in `Documentation_template.md` inside it before the final hand-in)
* `<team>_models.tar.gz` — trained models (inference-only reproduction, see README)
* `reports/`, `logs/` — validation numbers and full logs